# Experiment 5: SDTM Conversion (Python)
**Course:** 26AI2003 – Statistical Programming for Clinical Data Analytics

## Aim
To convert raw clinical data from multiple tables into standardized SDTM (Study Data Tabulation Model) datasets, preparing the data for regulatory submission and analysis.

## Description
Five raw source files are converted into five SDTM domains. Each domain gets the standard variables (`STUDYID`, `DOMAIN`, `USUBJID`, a sequence number `XXSEQ`, an ISO 8601 date `XXSTDTC`, and so on). Vital signs and labs arrive in **wide** format (one column per test) and are reshaped into **long** format (one row per test per visit). Every step is then checked against the SDTM validation checklist.

| Raw file | SDTM domain | Rows per subject | Main work |
|---|---|---|---|
| `raw_demographics.csv` | **DM** – Demographics | 1 | rename, build `USUBJID`, ISO dates |
| `raw_vitals.csv` | **VS** – Vital Signs | 1 per test per visit | wide → long, test codes, units |
| `raw_labs.csv` | **LB** – Laboratory | 1 per test per visit | wide → long, test codes, units |
| `raw_adverse_events.csv` | **AE** – Adverse Events | 1 per event | rename, ISO dates |
| `raw_exposure.csv` | **EX** – Exposure | 1 per drug | rename, ISO dates |

**Needs:** `pandas`. Keep the five raw CSV files in the same folder as this notebook.

*Note on names:* variable names follow your lab sheet (`VSRESN`, `VSUNIT`, `VSSTDTC`, `EXDRUG` …). The official CDISC names differ slightly (for example `VSSTRESN`, `VSDTC`, `EXTRT`).

## Step 1: Import library and load the raw data
`SUBJID` is read as text so that `001` keeps its leading zeros.

In [ ]:
import pandas as pd

STUDYID = "STUDY01"

raw_dm = pd.read_csv("raw_demographics.csv",   dtype={"SUBJID": str})
raw_vs = pd.read_csv("raw_vitals.csv",         dtype={"SUBJID": str})
raw_lb = pd.read_csv("raw_labs.csv",           dtype={"SUBJID": str})
raw_ae = pd.read_csv("raw_adverse_events.csv", dtype={"SUBJID": str})
raw_ex = pd.read_csv("raw_exposure.csv",       dtype={"SUBJID": str})

for name, d in [("DM", raw_dm), ("VS", raw_vs), ("LB", raw_lb), ("AE", raw_ae), ("EX", raw_ex)]:
    print(name, "raw shape:", d.shape)
raw_vs.head()

## Step 2: Helper functions
Two small helpers are reused for every domain:
- `iso_date` converts a raw date to ISO 8601 (`YYYY-MM-DD`). The raw files use two formats: `MM/DD/YYYY` and `DD-Mon-YYYY`.
- `add_seq` numbers the rows 1, 2, 3 … **within each subject** (the `XXSEQ` variable).

In [ ]:
def iso_date(column, fmt):
    return pd.to_datetime(column, format=fmt).dt.strftime("%Y-%m-%d")

def add_seq(data, seq_name, sort_cols):
    data = data.sort_values(sort_cols).reset_index(drop=True)
    data[seq_name] = data.groupby("USUBJID").cumcount() + 1
    return data

## Step 3: DM – Demographics (one row per subject)

In [ ]:
dm = pd.DataFrame({
    "STUDYID": STUDYID,
    "DOMAIN":  "DM",
    "USUBJID": STUDYID + "-" + raw_dm["SUBJID"],
    "SUBJID":  raw_dm["SUBJID"],
    "SITEID":  raw_dm["SITEID"],
    "AGE":     raw_dm["AGE"],
    "AGEU":    "YEARS",
    "SEX":     raw_dm["SEX"],
    "RACE":    raw_dm["RACE"],
    "ARM":     raw_dm["ARM"],
    "RFSTDTC": iso_date(raw_dm["FIRST_DOSE_DATE"], "%m/%d/%Y"),
    "RFENDTC": iso_date(raw_dm["LAST_DOSE_DATE"],  "%m/%d/%Y"),
})
dm = add_seq(dm, "DMSEQ", ["USUBJID"])
dm = dm[["STUDYID", "DOMAIN", "USUBJID", "DMSEQ", "SUBJID", "SITEID", "AGE", "AGEU",
         "SEX", "RACE", "ARM", "RFSTDTC", "RFENDTC"]]
print("DM rows:", len(dm))
dm.head()

## Step 4: VS – Vital Signs (wide → long)
The raw file has one row per visit with three vital-sign columns. `pd.melt` turns each column into its own row, so SDTM gets **one row per vital sign per visit**. A mapping then adds the standard test name and unit.

In [ ]:
vs_long = pd.melt(
    raw_vs,
    id_vars=["SUBJID", "VISIT", "VISITNUM", "VISIT_DATE"],
    value_vars=["SYSBP", "DIABP", "HEARTRATE"],
    var_name="VSTESTCD",
    value_name="VSRESN",
)

vs_test = {"SYSBP": "Systolic Blood Pressure",
           "DIABP": "Diastolic Blood Pressure",
           "HEARTRATE": "Heart Rate"}
vs_unit = {"SYSBP": "mmHg", "DIABP": "mmHg", "HEARTRATE": "beats/min"}

vs_long["STUDYID"] = STUDYID
vs_long["DOMAIN"]  = "VS"
vs_long["USUBJID"] = STUDYID + "-" + vs_long["SUBJID"]
vs_long["VSTEST"]  = vs_long["VSTESTCD"].map(vs_test)
vs_long["VSUNIT"]  = vs_long["VSTESTCD"].map(vs_unit)
vs_long["VSSTDTC"] = iso_date(vs_long["VISIT_DATE"], "%d-%b-%Y")

vs = add_seq(vs_long, "VSSEQ", ["USUBJID", "VISITNUM", "VSTESTCD"])
vs = vs[["STUDYID", "DOMAIN", "USUBJID", "VSSEQ", "VSTESTCD", "VSTEST", "VSRESN",
         "VSUNIT", "VISITNUM", "VISIT", "VSSTDTC"]]
print("VS rows:", len(vs), "(= 40 subjects x 3 visits x 3 tests)")
vs.head(6)

## Step 5: LB – Laboratory (wide → long)
Same idea as VS. A lab test that was not done (blank result) is **not** written to SDTM, so those rows are removed and counted.

In [ ]:
lb_long = pd.melt(
    raw_lb,
    id_vars=["SUBJID", "VISIT", "VISITNUM", "VISIT_DATE"],
    value_vars=["GLUCOSE", "CHOLESTEROL"],
    var_name="LBTEST_RAW",
    value_name="LBRESN",
)

lb_code = {"GLUCOSE": "GLUC", "CHOLESTEROL": "CHOL"}
lb_name = {"GLUC": "Glucose", "CHOL": "Cholesterol"}

lb_long["LBTESTCD"] = lb_long["LBTEST_RAW"].map(lb_code)
lb_long["LBTEST"]   = lb_long["LBTESTCD"].map(lb_name)
lb_long["LBUNIT"]   = "mg/dL"

n_missing = lb_long["LBRESN"].isna().sum()
lb_long = lb_long.dropna(subset=["LBRESN"])
print("Lab results missing in raw data and removed:", n_missing)

lb_long["STUDYID"] = STUDYID
lb_long["DOMAIN"]  = "LB"
lb_long["USUBJID"] = STUDYID + "-" + lb_long["SUBJID"]
lb_long["LBSTDTC"] = iso_date(lb_long["VISIT_DATE"], "%d-%b-%Y")

lb = add_seq(lb_long, "LBSEQ", ["USUBJID", "VISITNUM", "LBTESTCD"])
lb = lb[["STUDYID", "DOMAIN", "USUBJID", "LBSEQ", "LBTESTCD", "LBTEST", "LBRESN",
         "LBUNIT", "VISITNUM", "VISIT", "LBSTDTC"]]
print("LB rows:", len(lb))
lb.head(6)

## Step 6: AE – Adverse Events (one row per event)

In [ ]:
ae = pd.DataFrame({
    "STUDYID": STUDYID,
    "DOMAIN":  "AE",
    "USUBJID": STUDYID + "-" + raw_ae["SUBJID"],
    "AETERM":  raw_ae["AE_TERM"].str.upper(),
    "AESTDTC": iso_date(raw_ae["AE_START"], "%d-%b-%Y"),
    "AEENDTC": iso_date(raw_ae["AE_END"],   "%d-%b-%Y"),
    "AESEV":   raw_ae["SEVERITY"],
    "AEREL":   raw_ae["RELATIONSHIP"],
})
ae = add_seq(ae, "AESEQ", ["USUBJID", "AESTDTC"])
ae = ae[["STUDYID", "DOMAIN", "USUBJID", "AESEQ", "AETERM", "AESTDTC", "AEENDTC", "AESEV", "AEREL"]]
print("AE rows:", len(ae))
ae.head()

## Step 7: EX – Exposure (one row per drug)

In [ ]:
ex = pd.DataFrame({
    "STUDYID": STUDYID,
    "DOMAIN":  "EX",
    "USUBJID": STUDYID + "-" + raw_ex["SUBJID"],
    "EXDRUG":  raw_ex["DRUG"],
    "EXDOSE":  raw_ex["DOSE"],
    "EXUNIT":  raw_ex["DOSE_UNIT"],
    "EXSTDTC": iso_date(raw_ex["START_DATE"], "%m/%d/%Y"),
    "EXENDTC": iso_date(raw_ex["END_DATE"],   "%m/%d/%Y"),
})
ex = add_seq(ex, "EXSEQ", ["USUBJID", "EXSTDTC"])
ex = ex[["STUDYID", "DOMAIN", "USUBJID", "EXSEQ", "EXDRUG", "EXDOSE", "EXUNIT", "EXSTDTC", "EXENDTC"]]
print("EX rows:", len(ex))
ex.head()

## Step 8: Validation against the SDTM checklist
One function runs the checklist items on every domain and prints PASS or FAIL.

In [ ]:
domains = {
    "DM": (dm, "DMSEQ", None),
    "VS": (vs, "VSSEQ", "VSSTDTC"),
    "LB": (lb, "LBSEQ", "LBSTDTC"),
    "AE": (ae, "AESEQ", "AESTDTC"),
    "EX": (ex, "EXSEQ", "EXSTDTC"),
}
iso_pattern = r"^\d{4}-\d{2}-\d{2}$"

def show(label, ok):
    print("   ", "PASS" if ok else "FAIL", "-", label)

for code_, (d, seq, dtc) in domains.items():
    print(code_, "-", len(d), "rows")
    show("DOMAIN variable equals '%s'" % code_, (d["DOMAIN"] == code_).all())
    show("no missing USUBJID / DOMAIN / sequence", d[["USUBJID", "DOMAIN", seq]].notna().all().all())
    show("%s is unique within each subject" % seq, not d.duplicated(["USUBJID", seq]).any())
    show("no duplicate records", not d.duplicated().any())
    if dtc:
        show("%s is ISO 8601 (YYYY-MM-DD)" % dtc, d[dtc].str.match(iso_pattern).all())
    resn = [c for c in d.columns if c.endswith("RESN")]
    for c in resn:
        show("%s is numeric with no blanks" % c, pd.api.types.is_numeric_dtype(d[c]) and d[c].notna().all())
    print()

In [ ]:
# Every subject in VS, LB, AE and EX must also exist in DM
print("Subjects in DM:", dm["USUBJID"].nunique())
for code_, d in [("VS", vs), ("LB", lb), ("AE", ae), ("EX", ex)]:
    missing = set(d["USUBJID"]) - set(dm["USUBJID"])
    print(f"{code_}: subjects not found in DM =", len(missing))

# Long-format check: VS must have 3 rows per subject per visit
print()
print("VS rows per subject per visit (should all be 3):",
      vs.groupby(["USUBJID", "VISITNUM"]).size().unique())

### Timeline check: every date must fall between the subject's first and last dose

In [ ]:
dates = {"VS": (vs, "VSSTDTC"), "LB": (lb, "LBSTDTC"), "AE": (ae, "AESTDTC"), "EX": (ex, "EXSTDTC")}

for code_, (d, dtc) in dates.items():
    m = d.merge(dm[["USUBJID", "RFSTDTC", "RFENDTC"]], on="USUBJID")
    # ISO dates compare correctly as text
    bad = m[(m[dtc] < m["RFSTDTC"]) | (m[dtc] > m["RFENDTC"])]
    print(code_, ": dates outside the study timeline =", len(bad))
    if len(bad) > 0:
        print(bad[["USUBJID", dtc, "RFSTDTC", "RFENDTC"]].to_string(index=False))

## Step 9: Traceability – raw variable → SDTM variable
| SDTM variable | Comes from (raw file → column) | Derivation |
|---|---|---|
| `USUBJID` (all domains) | every raw file → `SUBJID` | `"STUDY01-"` + SUBJID |
| `DM.RFSTDTC`, `DM.RFENDTC` | demographics → `FIRST_DOSE_DATE`, `LAST_DOSE_DATE` | MM/DD/YYYY → ISO 8601 |
| `DM.AGEU` | none | constant "YEARS" |
| `VS.VSTESTCD`, `VS.VSRESN` | vitals → column names `SYSBP`, `DIABP`, `HEARTRATE` and their values | `pd.melt` (wide → long) |
| `VS.VSTEST`, `VS.VSUNIT` | none | mapping from `VSTESTCD` |
| `VS.VSSTDTC` | vitals → `VISIT_DATE` | DD-Mon-YYYY → ISO 8601 |
| `LB.LBTESTCD`, `LB.LBRESN` | labs → `GLUCOSE`, `CHOLESTEROL` | `pd.melt`; GLUCOSE→GLUC, CHOLESTEROL→CHOL; blanks removed |
| `LB.LBSTDTC` | labs → `VISIT_DATE` | DD-Mon-YYYY → ISO 8601 |
| `AE.AETERM`, `AESEV`, `AEREL` | adverse events → `AE_TERM`, `SEVERITY`, `RELATIONSHIP` | upper-case / renamed |
| `AE.AESTDTC`, `AEENDTC` | adverse events → `AE_START`, `AE_END` | DD-Mon-YYYY → ISO 8601 |
| `EX.EXDRUG`, `EXDOSE`, `EXUNIT` | exposure → `DRUG`, `DOSE`, `DOSE_UNIT` | renamed |
| `EX.EXSTDTC`, `EXENDTC` | exposure → `START_DATE`, `END_DATE` | MM/DD/YYYY → ISO 8601 |
| `DMSEQ`, `VSSEQ`, `LBSEQ`, `AESEQ`, `EXSEQ` | none | row number within subject after sorting |

## Step 10: Save the SDTM datasets and summarise

In [ ]:
dm.to_csv("dm.csv", index=False)
vs.to_csv("vs.csv", index=False)
lb.to_csv("lb.csv", index=False)
ae.to_csv("ae.csv", index=False)
ex.to_csv("ex.csv", index=False)

summary = pd.DataFrame({
    "Domain":   ["DM", "VS", "LB", "AE", "EX"],
    "Raw rows": [len(raw_dm), len(raw_vs), len(raw_lb), len(raw_ae), len(raw_ex)],
    "SDTM rows": [len(dm), len(vs), len(lb), len(ae), len(ex)],
    "Variables": [dm.shape[1], vs.shape[1], lb.shape[1], ae.shape[1], ex.shape[1]],
})
print("Saved: dm.csv, vs.csv, lb.csv, ae.csv, ex.csv")
summary